OBJECTIVE: Merge 116 csv files so that we have all of the accidents from 2013 to 2022 inclusive in one csv file.
1) Import libraries to deal with numbers, dataframes and file management 

In [1]:
import numpy as np
import pandas as pd
import glob
from pathlib import Path

2) Make a list of all csv files to use:

In [2]:
folder_path = r"C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months"
list_of_files = glob.glob(folder_path + "/*.csv")
file_pattern = "C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months/*.csv"

3) Check that each file has the correct number (37) columns:
For each file in the list, it is read as a csv file into a pandas dataframe.
Then we check if the dataframe does not have 37 columns, in which case the filename and actual number of columns is printed.

In [3]:
for file in list_of_files:
    df = pd.read_csv(file, sep=';', quotechar='"', decimal=",", header=1)
    if df.shape[1] != 37:
        print(f"{file} has {df.shape[1]} columns")

C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2015-10.csv has 38 columns
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2018-06.csv has 38 columns
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2018-07.csv has 38 columns
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2018-08.csv has 38 columns
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2018-09.csv has 38 columns
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2018-10.csv has 38 columns
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2018-11.csv has 38 columns
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2018-12.csv has 38 columns
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2019-01.csv has 38 columns
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2019-02.csv has 38 columns
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2019-03.csv has

There should be 37 columns in each file. 

38 columns are present in: 2015-10 and every file from 2018-06 to 2022-08 inclusive.

We list the column names for a sample file, 2022-01:

In [4]:
sample_file = "C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2022-01.csv"
df_sample_file = pd.read_csv(
    file, sep=';', quotechar='"', decimal=",", header=0)
df_sample_file.head(2).T

,0,1
Protocollo,6071051,6071051
Gruppo,6,6
DataOraIncidente,01/08/2022 01:45:00,01/08/2022 01:45:00
Localizzazione1,Strada Urbana,Strada Urbana
STRADA1,PIAZZA VITTORIO EMANUELE II,PIAZZA VITTORIO EMANUELE II
Localizzazione2,all'intersezione con,all'intersezione con
STRADA2,VIA CONTE VERDE,VIA CONTE VERDE
Strada02,VIA CONTE VERDE,VIA CONTE VERDE
Chilometrica,NaN,NaN
DaSpecificare,NaN,NaN


4) There is an extra unwanted column (index 37), called 'Unnamed: 37' in this case.
Where the file has 38 columns, we check what is in that last 'extra' column.

This code checks every file, finds those with a column called 'Unnamed: 37' and non null items in that column.
The non-null items are printed out, together with the row numbers and the filename. 

In [5]:
for file in list_of_files:
    items = []
    df = pd.read_csv(file, encoding='latin1', sep=';',
                     quotechar='"', decimal=",", header=0)
    if ('Unnamed: 37' in df.columns) and df.iloc[1:, 37].notna().any():
        print(f"{file}: has 'Unnamed: 37' column - ERROR")
        location_items = df[df['Unnamed: 37'].notna()].index.tolist()
        items = df['Unnamed: 37'].dropna().tolist()
        print(
            f"Non-null items in 38th column: {items} on rows {location_items}")

C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2015-10.csv: has 'Unnamed: 37' column - ERROR
Non-null items in 38th column: ['Inesploso', 'Inesploso'] on rows [132, 133]
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2022-06.csv: has 'Unnamed: 37' column - ERROR
Non-null items in 38th column: ['Inesploso'] on rows [2380]


For the 2015-10 and 2022-06 files, rows 132-133 and 2380 respectively contain the value 'Inesploso', which is from the column to the left, 'Airbag'. It seems that the row data is not correctly aligned with the column headers. 

However, there may be other rows which are misaligned with the column headers. The check above would catch errors where the 37th column has a non-null value, but not those where the 37th column doesn't. 

In [6]:
natura_incidente_options = set()
target_column = 'NaturaIncidente'

for file in list_of_files:
    df = pd.read_csv(file, encoding='latin1', sep=';',
                     quotechar='"', decimal=",", header=0)
    natura_incidente_options.update(df[target_column].dropna().unique())

natura_incidente_list = sorted(natura_incidente_options)
print(natura_incidente_list)

[' altezza civico 19', ' spartitraffico per inversione', 'Fuoriuscita dalla sede stradale', 'Infortunio per caduta del veicolo', 'Infortunio per sola frenata improvvisa', 'Investimento di pedone', 'Ribaltamento senza urto contro ostacolo fisso', 'Scontro frontale fra veicoli in marcia', 'Scontro frontale/laterale DX fra veicoli in marcia', 'Scontro frontale/laterale SX fra veicoli in marcia', 'Scontro laterale fra veicoli in marcia', 'Tamponamento', 'Tamponamento Multiplo', 'Veicoli in marcia contro veicoli fermi', 'Veicoli in marcia contro veicolo fermo', 'Veicolo in marcia che urta buche nella carreggiata', 'Veicolo in marcia contro animale', 'Veicolo in marcia contro ostacolo accidentale', 'Veicolo in marcia contro ostacolo fisso', 'Veicolo in marcia contro treno', 'Veicolo in marcia contro veicoli fermi', 'Veicolo in marcia contro veicoli in arresto', 'Veicolo in marcia contro veicoli in sosta', 'Veicolo in marcia contro veicolo arresto', 'Veicolo in marcia contro veicolo fermo', '

Each option for 'NaturaIncidente' should describe the type of accident and start with a capital. The values which should not be there are the first two in this list, which describe the location of the accident, not the type. To correct this list, we should cut the first two values:

In [7]:
natura_incidente_list = sorted(natura_incidente_list)[2:]
natura_incidente_list

['Fuoriuscita dalla sede stradale',
 'Infortunio per caduta del veicolo',
 'Infortunio per sola frenata improvvisa',
 'Investimento di pedone',
 'Ribaltamento senza urto contro ostacolo fisso',
 'Scontro frontale fra veicoli in marcia',
 'Scontro frontale/laterale DX fra veicoli in marcia',
 'Scontro frontale/laterale SX fra veicoli in marcia',
 'Scontro laterale fra veicoli in marcia',
 'Tamponamento',
 'Tamponamento Multiplo',
 'Veicoli in marcia contro veicoli fermi',
 'Veicoli in marcia contro veicolo fermo',
 'Veicolo in marcia che urta buche nella carreggiata',
 'Veicolo in marcia contro animale',
 'Veicolo in marcia contro ostacolo accidentale',
 'Veicolo in marcia contro ostacolo fisso',
 'Veicolo in marcia contro treno',
 'Veicolo in marcia contro veicoli fermi',
 'Veicolo in marcia contro veicoli in arresto',
 'Veicolo in marcia contro veicoli in sosta',
 'Veicolo in marcia contro veicolo arresto',
 'Veicolo in marcia contro veicolo fermo',
 'Veicolo in marcia contro veicolo 

Now we have a list of the correct options for the 'NaturaIncidente' column. We repeat the process for the next column, 'particolaritastrade' which describes the road features. 

In [8]:
particolarita_strade_options = set()
target_column = 'particolaritastrade'

for file in list_of_files:
    df = pd.read_csv(file, encoding='latin1', sep=';',
                     quotechar='"', decimal=",", header=0)
    particolarita_strade_options.update(df[target_column].dropna().unique())

particolarita_strade_list = sorted(particolarita_strade_options)
print(particolarita_strade_list)

['Cavalcavia', 'Cunetta', 'Curva a visuale libera', 'Curva senza visuale libera', 'Dosso', 'Galleria illuminata', 'Galleria non illuminata', 'Incrocio', 'Intersezione non regolata/non segnalata', 'Intersezione regolata dal vigile', 'Intersezione semaforizzata', 'Intersezione stradale segnalata', 'Passaggio a livello (custodito)', 'Passaggio a livello (incustodito)', 'Pendenza', 'Pianeggiante', 'Rettilineo', 'Rotatoria', 'Scontro frontale/laterale SX fra veicoli in marcia', 'Scontro laterale fra veicoli in marcia', 'Sottopasso illuminato', 'Sottopasso non illuminato', 'Strettoia']


In [9]:
len(particolarita_strade_list)

23

Now we will check if any values in particolarita_strade_list are in natura_incidente_list and remove them, leaving a 'clean' list of options for particolarita_strade (type of road):

In [10]:
elements_to_remove_from_particolarita_strade = set(natura_incidente_list)

particolarita_strade_list_clean = []
for i in particolarita_strade_list:
    if i not in elements_to_remove_from_particolarita_strade:
        particolarita_strade_list_clean.append(i)

particolarita_strade_list = particolarita_strade_list_clean
print(particolarita_strade_list)

['Cavalcavia', 'Cunetta', 'Curva a visuale libera', 'Curva senza visuale libera', 'Dosso', 'Galleria illuminata', 'Galleria non illuminata', 'Incrocio', 'Intersezione non regolata/non segnalata', 'Intersezione regolata dal vigile', 'Intersezione semaforizzata', 'Intersezione stradale segnalata', 'Passaggio a livello (custodito)', 'Passaggio a livello (incustodito)', 'Pendenza', 'Pianeggiante', 'Rettilineo', 'Rotatoria', 'Sottopasso illuminato', 'Sottopasso non illuminato', 'Strettoia']


In [11]:
len(particolarita_strade_list)

21

There were two items in the 'type of road' list which were actually from the 'type of accident' column. Now we have a clean list of the options for both 'type of accident' and 'type of road'. 

Now we can search all files to determine how many rows of data there are, where values are out of place (using the particolaritastrade and NaturaIncidente columns), the total number of values (to make sure there is nothing else out of place) and the row numbers where the errors were found.

In [12]:
for file in list_of_files:
    df = pd.read_csv(file, encoding='latin1', sep=';',
                     quotechar='"', decimal=",", header=0)
    protocollo_count = len(df['Protocollo'])

    if 'particolaritastrade' not in df.columns:
        print(f"{file}: particolaritastrade column not found")
        continue
    mask = df['particolaritastrade'].isin(natura_incidente_list)

    if mask.any():
        print("\n", file, df[['particolaritastrade']
                             ].value_counts(dropna=False))
        print("\nValues in Protocollo column: ", protocollo_count)
        print("Dataframe length:            ", len(df))
        incorrect_count = int(mask.sum())
        print("\nItems in incorrect column: ", incorrect_count)
        row_numbers_wrong = df[mask].index.tolist()
        print("Row numbers with incorrect values:", row_numbers_wrong)


 C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2015-10.csv particolaritastrade                    
Rettilineo                                 3262
Incrocio                                   1238
Intersezione semaforizzata                  664
Curva a visuale libera                      356
Intersezione stradale segnalata             294
Curva senza visuale libera                  181
Rotatoria                                   158
Intersezione non regolata/non segnalata      55
Pendenza                                     54
Galleria illuminata                          35
Sottopasso illuminato                        14
Cavalcavia                                   13
Pianeggiante                                  5
Strettoia                                     4
Scontro laterale fra veicoli in marcia        2
Sottopasso non illuminato                     2
Name: count, dtype: int64

Values in Protocollo column:  6337
Dataframe length:             6337

Items in incorr

Misaligned values from column index 10 exist in the particolarita strade column (index 11) for files:
2015-10, rows 132 and 133
2022-06, rows 2379 to 2385 inclusive

However, if the 'NaturaIncidente' column has blank values where the rows are misaligned with the headers, the check above would not detect them. We can check for misaligned columns again, this time checking if the 'Traffico' column's values (index 17) have ended up in the 'Visibilita' column (index 18):

In [13]:
traffico_values = ['Intenso', 'Scarso', 'Normale']

for file in list_of_files:
    df = pd.read_csv(file, sep=';', quotechar='"', decimal=",", header=0)
    protocollo_count = len(df['Protocollo'])
    if 'Visibilita' not in df.columns:
        print(f"{file}: Visibilita column not found.")
        continue
    mask_traffico = df['Visibilita'].isin(traffico_values)

    if mask_traffico.any():
        print("\n", file, df[['Visibilita']
                             ].value_counts(dropna=False))
        print("\nValues in Protocollo column: ", protocollo_count)
        print("Dataframe length:            ", len(df))
        incorrect_count = int(mask_traffico.sum())
        print("\n Items in incorrect column: ", incorrect_count)
        row_numbers_vis_wrong = df[mask_traffico].index.tolist()
        print("Row numbers with incorrect values:", row_numbers_vis_wrong)


 C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2015-10.csv Visibilita   
Buona            5051
Sufficiente      1149
Insufficiente     135
Normale             2
Name: count, dtype: int64

Values in Protocollo column:  6337
Dataframe length:             6337

 Items in incorrect column:  2
Row numbers with incorrect values: [132, 133]

 C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2022-06.csv Visibilita   
Buona            5018
Sufficiente       621
NaN               102
Insufficiente      53
Normale             7
Name: count, dtype: int64

Values in Protocollo column:  5801
Dataframe length:             5801

 Items in incorrect column:  7
Row numbers with incorrect values: [2379, 2380, 2381, 2382, 2383, 2384, 2385]


Before dropping the unwanted 38th column, we need to first realign the data so that the data shifts one column left:

File 2015-10 rows 132 and 133 
File 2022-06 rows 2379 to 2385 inclusive 

Let's check if column index 8 and 9 are empty (so that the items in column index 10 can move left by one without losing data)

In [20]:
first_file = "C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months/2015-10.csv"

df_first_file = pd.read_csv(
    first_file, encoding='latin1', sep=';', quotechar='"', decimal=",", header=0)

df_first_file.iloc[132:134, 8:11]

,Chilometrica,DaSpecificare,NaturaIncidente
132,NaN,altezza civ. 62,spartitraffico per inversione
133,NaN,altezza civ. 62,spartitraffico per inversione


We will lose data if we shift column index 10 (NaturaIncidente) into column 9 (DaSpecificare). 
But as column index 8 has null values, we can move 9 and 10 into 8 and 9 without losing data. 

In [21]:
df_first_file.iloc[132:134, 8:37] = df_first_file.iloc[132:134, 9:38].values
df_first_file.to_csv(first_file, encoding='latin1', sep=';',
                     index=False, quotechar='"', decimal=",")

C:\Users\lucyq\AppData\Local\Temp\ipykernel_19032\3256147323.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[2.0, 2.0]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_first_file.iloc[132:134, 8:37] = df_first_file.iloc[132:134, 9:38].values
C:\Users\lucyq\AppData\Local\Temp\ipykernel_19032\3256147323.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['1', '2']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_first_file.iloc[132:134, 8:37] = df_first_file.iloc[132:134, 9:38].values


Now recheck the first file, 2015-10:

In [22]:
first_file = "C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months/2015-10.csv"

df_first_file = pd.read_csv(
    first_file, encoding='latin1', sep=';', quotechar='"', decimal=",", header=0)

df_first_file.iloc[132:134, 8:11]

,Chilometrica,DaSpecificare,NaturaIncidente
132,altezza civ. 62,spartitraffico per inversione,Scontro laterale fra veicoli in marcia
133,altezza civ. 62,spartitraffico per inversione,Scontro laterale fra veicoli in marcia


Although there are warnings about the types in each column (which we can fix later), the shift is complete. Checking the last few columns:

In [23]:
df_first_file.iloc[132:134, 30:38]

,TipoPersona,Sesso,Tipolesione,Deceduto,DecedutoDopo,CinturaCascoUtilizzato,Airbag,Unnamed: 37
132,Conducente,M,Illeso,0,NaN,Non accertato,Inesploso,Inesploso
133,Conducente,M,Illeso,0,NaN,Non accertato,Inesploso,Inesploso


The values from the Unnamed column have successfully been copied into the column to the left. The unnamed: 37 column remains, which will be cut later.

Now we can repeat the process for the 2022-06 file. 
We check the columns to the left to see what is there first. 

In [25]:
second_file = "C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months/2022-06.csv"

df_second_file = pd.read_csv(
    second_file, encoding='latin1', sep=';', quotechar='"', decimal=",", header=0)

df_second_file.iloc[2379:2386, 8:11]

,Chilometrica,DaSpecificare,NaturaIncidente
2379,NaN,prossimitÃ via Campofranco,altezza civico 19
2380,NaN,prossimitÃ via Campofranco,altezza civico 19
2381,NaN,prossimitÃ via Campofranco,altezza civico 19
2382,NaN,prossimitÃ via Campofranco,altezza civico 19
2383,NaN,prossimitÃ via Campofranco,altezza civico 19
2384,NaN,prossimitÃ via Campofranco,altezza civico 19
2385,NaN,prossimitÃ via Campofranco,altezza civico 19


We will lose data if we shift column index 10 (NaturaIncidente) into column 9 (DaSpecificare). 
But as column index 8 has null values, we can move 9 and 10 into 8 and 9 without losing data. 

In [ ]:
df_second_file.iloc[2379:2386,
                    8:37] = df_second_file.iloc[2379:2386, 9:38].values


df_second_file.to_csv(second_file, encoding='latin1', sep=';', index=False,






















                      quotechar='"', decimal=",")

C:\Users\lucyq\AppData\Local\Temp\ipykernel_19032\3055008235.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_second_file.iloc[2379:2386, 8:37] = df_second_file.iloc[2379:2386, 9:38].values
C:\Users\lucyq\AppData\Local\Temp\ipykernel_19032\3055008235.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['1', '1', '2', '2', '2', '2', '3']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_second_file.iloc[2379:2386, 8:37] = df_second_file.iloc[2379:2386, 9:38].values


Again, we have warnings, but the shift has taken place successfully. We will alter the types later. Let's check the column index 8 to 11 to be sure:

In [27]:
df_second_file.iloc[2379:2386, 8:11]

,Chilometrica,DaSpecificare,NaturaIncidente
2379,prossimitÃ via Campofranco,altezza civico 19,Scontro frontale/laterale SX fra veicoli in ma...
2380,prossimitÃ via Campofranco,altezza civico 19,Scontro frontale/laterale SX fra veicoli in ma...
2381,prossimitÃ via Campofranco,altezza civico 19,Scontro frontale/laterale SX fra veicoli in ma...
2382,prossimitÃ via Campofranco,altezza civico 19,Scontro frontale/laterale SX fra veicoli in ma...
2383,prossimitÃ via Campofranco,altezza civico 19,Scontro frontale/laterale SX fra veicoli in ma...
2384,prossimitÃ via Campofranco,altezza civico 19,Scontro frontale/laterale SX fra veicoli in ma...
2385,prossimitÃ via Campofranco,altezza civico 19,Scontro frontale/laterale SX fra veicoli in ma...


And check the last few columns:

In [28]:
df_second_file.iloc[2379:2386, 30:38]

,TipoPersona,Sesso,Tipolesione,Deceduto,DecedutoDopo,CinturaCascoUtilizzato,Airbag,Unnamed: 37
2379,Conducente,M,Illeso,0,NaN,NaN,NaN,NaN
2380,Passeggero,M,Illeso,0,NaN,Non accertato,Inesploso,Inesploso
2381,Conducente,M,Illeso,0,NaN,NaN,NaN,NaN
2382,Passeggero,M,Illeso,0,NaN,Non accertato,NaN,NaN
2383,Passeggero,F,Illeso,0,NaN,NaN,NaN,NaN
2384,Passeggero,M,Illeso,0,NaN,NaN,NaN,NaN
2385,Conducente,M,Illeso,0,NaN,NaN,NaN,NaN


Again, we will need to cut the last column, Unnamed:37
We can do this for every file which has a column of that name. 

In [29]:
for file in list_of_files:
    df = pd.read_csv(file, encoding='latin1', sep=';',
                     quotechar='"', decimal=",", header=0)
    if df.shape[1] == 38:
        df = df.drop(columns=['Unnamed: 37'])
        df.to_csv(file, encoding='latin1', sep=';',
                  quotechar='"', decimal=",", index=False)
        print(f"{file}: Column dropped. New shape {df.shape[1]}")
    else:
        print(f"{file}: No columns removed")

C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-01.csv: No columns removed
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-02.csv: No columns removed
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-03.csv: No columns removed
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-04.csv: No columns removed
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-05.csv: No columns removed
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-06.csv: No columns removed
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-07.csv: No columns removed
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-08.csv: No columns removed
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-09.csv: No columns removed
C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-10.csv: No columns removed
C:/Users/lucyq/Dropbox/AMDP/TH

Now all the files have 37 columns. 

In [32]:
file_a = "C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months/2013-01.csv"
df_a = pd.read_csv(file_a, encoding='latin1', sep=';',
                   quotechar='"', decimal=",", header=0)

# Create type dictionary with string representations
dtype_dict = {col: str(dtype) for col, dtype in df_a.dtypes.items()}

print(f"Type dictionary from {file_a}:")
dtype_dict

Type dictionary from C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months/2013-01.csv:


{'Protocollo': 'int64',
 'Gruppo': 'int64',
 'DataOraIncidente': 'object',
 'Localizzazione1': 'object',
 'STRADA1': 'object',
 'Localizzazione2': 'object',
 'STRADA2': 'object',
 'Strada02': 'object',
 'Chilometrica': 'object',
 'DaSpecificare': 'object',
 'NaturaIncidente': 'object',
 'particolaritastrade': 'object',
 'TipoStrada': 'object',
 'FondoStradale': 'object',
 'Pavimentazione': 'object',
 'Segnaletica': 'object',
 'CondizioneAtmosferica': 'object',
 'Traffico': 'object',
 'Visibilita': 'object',
 'Illuminazione': 'object',
 'NUM_FERITI': 'int64',
 'NUM_RISERVATA': 'int64',
 'NUM_MORTI': 'int64',
 'NUM_ILLESI': 'int64',
 'Longitude': 'float64',
 'Latitude': 'float64',
 'Confermato': 'int64',
 'Progressivo': 'float64',
 'TipoVeicolo': 'object',
 'StatoVeicolo': 'object',
 'TipoPersona': 'object',
 'Sesso': 'object',
 'Tipolesione': 'object',
 'Deceduto': 'int64',
 'DecedutoDopo': 'object',
 'CinturaCascoUtilizzato': 'object',
 'Airbag': 'object'}

Most columns are already of the correct type in this file. 

FLOAT - SHOULD BE INTEGER: 
'Progressivo' is a random numbering used to identify the different parties in each accident. Pedestrians have no number, after which each vehicle involved takes a random number. This should be an integer.  

OBJECT - COULD BE CATEGORY?
Most of the columns contain 'object' data. Using 'category' would be more efficient as there are a small number of unique values in each column (up to around max 30). The type 'category' means the df would store each unique value once, and use integer codes internally.

In [33]:
df_a = pd.read_csv(file, encoding='latin1', sep=';',
                   quotechar='"', decimal=",", header=0)
for col in df_a.select_dtypes(include='object'):
    unique_vals = df_a[col].nunique()
    total_vals = len(df_a)
    ratio = unique_vals / total_vals
    print(f"{col}: {unique_vals} unique ({ratio:.2%})")

    if unique_vals < 50 or ratio < 0.01:
        print(f" → ✅ Consider converting '{col}' to 'category'\n")

DataOraIncidente: 1052 unique (33.75%)
Localizzazione1: 9 unique (0.29%)
 → ✅ Consider converting 'Localizzazione1' to 'category'

STRADA1: 681 unique (21.85%)
Localizzazione2: 5 unique (0.16%)
 → ✅ Consider converting 'Localizzazione2' to 'category'

STRADA2: 360 unique (11.55%)
Strada02: 363 unique (11.65%)
Chilometrica: 298 unique (9.56%)
DaSpecificare: 267 unique (8.57%)
NaturaIncidente: 20 unique (0.64%)
 → ✅ Consider converting 'NaturaIncidente' to 'category'

particolaritastrade: 14 unique (0.45%)
 → ✅ Consider converting 'particolaritastrade' to 'category'

TipoStrada: 5 unique (0.16%)
 → ✅ Consider converting 'TipoStrada' to 'category'

FondoStradale: 4 unique (0.13%)
 → ✅ Consider converting 'FondoStradale' to 'category'

Pavimentazione: 6 unique (0.19%)
 → ✅ Consider converting 'Pavimentazione' to 'category'

Segnaletica: 5 unique (0.16%)
 → ✅ Consider converting 'Segnaletica' to 'category'

CondizioneAtmosferica: 5 unique (0.16%)
 → ✅ Consider converting 'CondizioneAtmosfer

Now we specify the desired column types:

In [35]:
columns_objects = ['DataOraIncidente',
                   'STRADA1',
                   'Localizzazione2',
                   'STRADA2',
                   'Strada02',
                   'Chilometrica',
                   'DaSpecificare']
columns_categories = ['Localizzazione1',
                      'NaturaIncidente',
                      'particolaritastrade',
                      'TipoStrada',
                      'FondoStradale',
                      'Pavimentazione',
                      'Segnaletica',
                      'CondizioneAtmosferica',
                      'Traffico',
                      'Visibilita',
                      'Illuminazione',
                      'TipoVeicolo',
                      'StatoVeicolo',
                      'TipoPersona',
                      'Sesso',
                      'Tipolesione',
                      'DecedutoDopo',
                      'CinturaCascoUtilizzato',
                      'Airbag']
columns_int64 = ['Protocollo',
                 'Gruppo',
                 'NUM_FERITI',
                 'NUM_RISERVATA',
                 'NUM_MORTI',
                 'NUM_ILLESI',
                 'Confermato',
                 'Progressivo',
                 'Deceduto']
columns_float64 = ['Longitude',
                   'Latitude']

We will change the object items into types, as appropriate: object (where many different values), category (where there are a limited number of different strings). We need to reset the dtype dictionary:

In [39]:
dtype_dict = {}

for col in (columns_objects or columns_categories):
    dtype_dict[col] = 'object'
for col in columns_int64:
    dtype_dict[col] = 'Int64'
for col in columns_float64:
    dtype_dict[col] = 'float64'

dtype_dict

{'DataOraIncidente': 'object',
 'STRADA1': 'object',
 'Localizzazione2': 'object',
 'STRADA2': 'object',
 'Strada02': 'object',
 'Chilometrica': 'object',
 'DaSpecificare': 'object',
 'Protocollo': 'Int64',
 'Gruppo': 'Int64',
 'NUM_FERITI': 'Int64',
 'NUM_RISERVATA': 'Int64',
 'NUM_MORTI': 'Int64',
 'NUM_ILLESI': 'Int64',
 'Confermato': 'Int64',
 'Progressivo': 'Int64',
 'Deceduto': 'Int64',
 'Longitude': 'float64',
 'Latitude': 'float64'}

In [40]:
columns_categories

['Localizzazione1',
 'NaturaIncidente',
 'particolaritastrade',
 'TipoStrada',
 'FondoStradale',
 'Pavimentazione',
 'Segnaletica',
 'CondizioneAtmosferica',
 'Traffico',
 'Visibilita',
 'Illuminazione',
 'TipoVeicolo',
 'StatoVeicolo',
 'TipoPersona',
 'Sesso',
 'Tipolesione',
 'DecedutoDopo',
 'CinturaCascoUtilizzato',
 'Airbag']

In [55]:
pip install pyarrow

   ---------------------------------------- 0.0/25.7 MB ? eta -:--:--
   ----------- ---------------------------- 7.6/25.7 MB 39.0 MB/s eta 0:00:01
   ---------------------------- ----------- 18.4/25.7 MB 44.6 MB/s eta 0:00:01
   ---------------------------------------  25.4/25.7 MB 46.0 MB/s eta 0:00:01
   ---------------------------------------- 25.7/25.7 MB 40.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [56]:
dataframes = []
output_file = 'merged_data.parquet'

for i, file in enumerate(list_of_files):
    try:
        print(f"Processing file {i+1}/{len(list_of_files)}: {file}")

        df = pd.read_csv(file,
                         dtype=str,
                         encoding='latin1',
                         sep=';',
                         quotechar='"',
                         header=0,
                         na_filter=False)

        # Convert categorical columns
        for col in columns_categories:
            if col in df.columns:
                df[col] = df[col].astype('category')

        # Convert float columns
        for col in columns_float64:
            if col in df.columns:
                try:
                    df[col] = df[col].replace(
                        ['nan', 'NaN', 'NULL', 'null', ''], pd.NA)
                    df[col] = df[col].astype(str).str.replace(
                        ',', '.', regex=False)
                    df[col] = df[col].replace('nan', pd.NA)
                    df[col] = pd.to_numeric(
                        df[col], errors='coerce').astype('float64')
                except Exception as e:
                    print(f"Warning: Could not convert {col} to float: {e}")

        # Convert integer columns to decimals first
        for col in columns_int64:
            if col in df.columns:
                try:
                    df[col] = df[col].replace(
                        ['nan', 'NaN', 'NULL', 'null', ''], pd.NA)
                    df[col] = df[col].astype(str).str.replace(
                        ',', '.', regex=False)
                    df[col] = df[col].replace('nan', pd.NA)
                    df[col] = pd.to_numeric(
                        df[col], errors='coerce').round().astype('Int64')
                except Exception as e:
                    print(f"Warning: Could not convert {col} to integer: {e}")

        dataframes.append(df)

    except Exception as e:
        print(f"Error processing {file}: {e}")
        continue  # Continue with next file even if this one fails

# Process results
if dataframes:
    # Concatenate all dataframes
    print("Concatenating all dataframes...")
    merged_df = pd.concat(dataframes, ignore_index=True)

    # Save as Parquet (data types preserved)
    print(f"Saving concatenated data to {output_file}")
    merged_df.to_parquet(output_file, index=False)

    # Save info about the data
    info_file = output_file.replace('.parquet', '_info.txt')
    with open(info_file, 'w') as f:
        f.write(f"Merged dataset info:\n")
        f.write(f"Total rows: {len(merged_df)}\n")
        f.write(f"Total columns: {len(merged_df.columns)}\n")
        f.write(f"Files processed: {len(dataframes)}\n\n")
        f.write("Data types:\n")
        f.write(str(merged_df.dtypes))
        f.write("\n\nMemory usage:\n")
        f.write(str(merged_df.memory_usage(deep=True)))

    print(
        f"Merge completed! Final dataset: {len(merged_df)} rows, {len(merged_df.columns)} columns")
else:
    print("No dataframes were successfully created")

Processing file 1/116: C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-01.csv
Processing file 2/116: C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-02.csv
Processing file 3/116: C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-03.csv
Processing file 4/116: C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-04.csv
Processing file 5/116: C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-05.csv
Processing file 6/116: C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-06.csv
Processing file 7/116: C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-07.csv
Processing file 8/116: C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-08.csv
Processing file 9/116: C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-09.csv
Processing file 10/116: C:/Users/lucyq/Dropbox/AMDP/THESIS/Roma_Capitale_OpenData/Months\2013-10.csv